# AI-Powered Food Demand Forecasting — Model Training Notebook

**Project:** BhojanSetu — AI-Powered Smart Food Waste Reduction and Sustainable Redistribution Ecosystem  
**Member 2:** Demand Forecasting + Surplus Prediction  

## Dataset Summary

| File | Rows | Key Columns |
|------|------|-------------|
| train.csv | 456,548 | id, week (1–145), center_id, meal_id, checkout_price, base_price, emailer_for_promotion, homepage_featured, **num_orders** |
| test.csv | 32,573 | Same minus num_orders (weeks 146–155) |
| meal_info.csv | 51 | meal_id, category, cuisine |
| fulfilment_center_info.csv | 77 | center_id, city_code, region_code, center_type, op_area |

- **Target variable:** `num_orders` (integer, weekly food orders per center-meal pair)
- **Time structure:** Integer week number (no calendar dates in raw data)
- **No missing values** in any core column
- **Model selected:** RandomForestRegressor — appropriate for tabular regression with mixed feature types, captures non-linearities, provides prediction intervals via tree variance


In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.getcwd()))  # project root

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

print('Libraries loaded')

## 1. Dataset Exploration

In [ ]:
train = pd.read_csv('../datasets/train.csv')
test  = pd.read_csv('../datasets/test.csv')
meal_info = pd.read_csv('../datasets/meal_info.csv')
center_info = pd.read_csv('../datasets/fulfilment_center_info.csv')

print('Train shape:', train.shape)
print('Test shape :', test.shape)
print('Weeks in train:', train['week'].min(), '-', train['week'].max())
print('Weeks in test :', test['week'].min(), '-', test['week'].max())
print()
print('Unique centers:', train['center_id'].nunique())
print('Unique meals  :', train['meal_id'].nunique())
print()
print('num_orders stats:')
print(train['num_orders'].describe())

In [ ]:
print('Null values in train:')
print(train.isnull().sum())
print()
print('meal_info categories:', meal_info['category'].unique())
print('meal_info cuisines  :', meal_info['cuisine'].unique())
print('center types        :', center_info['center_type'].unique())

In [ ]:
# Weekly aggregate demand trend
weekly = train.groupby('week')['num_orders'].sum().reset_index()
plt.figure(figsize=(12,4))
plt.plot(weekly['week'], weekly['num_orders'], linewidth=1)
plt.title('Total Weekly Orders Across All Centers')
plt.xlabel('Week')
plt.ylabel('Total num_orders')
plt.tight_layout()
plt.savefig('../reports/weekly_demand_trend.png', dpi=100)
plt.show()
print('Saved weekly_demand_trend.png')

## 2. Feature Engineering

Features used:
- `week`, `week_sin`, `week_cos`, `quarter`, `is_year_end` — time features
- `checkout_price`, `base_price`, `price_discount`, `discount_ratio` — price features
- `emailer_for_promotion`, `homepage_featured` — promotion features
- `center_id`, `meal_id` — identity features
- `category_enc`, `cuisine_enc`, `center_type_enc`, `op_area` — enrichment from lookup tables
- `event_flag`, `event_multiplier` — optional event calendar

## 3. Model Training

In [ ]:
from models.demand_forecast import DemandForecaster

# Train with 200 trees (production quality)
fc = DemandForecaster(n_estimators=200, random_state=42)
fc.train()

metrics = fc.get_metrics()
print('=== Evaluation Metrics ===')
for k, v in metrics.items():
    print('  %-20s: %s' % (k, round(v, 2) if isinstance(v, float) else v))

In [ ]:
print('=== Model Summary ===')
summary = fc.get_model_summary()
for k, v in summary.items():
    if k != 'feature_cols':
        print('  %-20s: %s' % (k, v))
print()
print('Features used:', summary['feature_cols'])

In [ ]:
# Feature importance
fi = fc.get_feature_importance()
print('Top 10 Features:')
for i, item in enumerate(fi[:10]):
    print('  %2d. %-25s %.5f' % (i+1, item['feature'], item['importance']))

## 4. Demand Forecasting — Example Outputs

In [ ]:
# Example: Next 7 weeks demand forecast for center 55, meal 1885
results = fc.forecast_next_n_weeks(center_id=55, meal_id=1885, n=7)
print('=== Next 7 Weeks Demand Forecast ===')
print('Center: 55  |  Meal: 1885')
print()
header = '%-6s %-18s %-12s %-12s %-12s'
print(header % ('Week', 'Predicted Demand', 'Lower Bound', 'Upper Bound', 'Std Dev'))
print('-' * 65)
for r in results:
    print('%-6d %-18.1f %-12.1f %-12.1f %-12.1f' % (
        r['week'], r['predicted_demand'], r['lower_bound'],
        r['upper_bound'], r['std_dev']))

In [ ]:
# Example with event calendar
events = [
    {'week': 147, 'event_name': 'Diwali', 'event_type': 'festival', 'multiplier': 1.3},
    {'week': 149, 'event_name': 'Exam Period', 'event_type': 'exam', 'multiplier': 0.7},
]
results_ev = fc.predict(55, 1885, weeks=list(range(146, 153)), events=events)
print('=== Forecast with Event Calendar ===')
for r in results_ev:
    flag = ''
    if r['event_multiplier'] != 1.0:
        flag = '<-- event (x%.1f)' % r['event_multiplier']
    print('Week %-3d: %.1f orders  %s' % (r['week'], r['predicted_demand'], flag))

## 5. Surplus Prediction Engine

In [ ]:
from models.surplus_predictor import SurplusPredictor

sp = SurplusPredictor()

# Example: Single surplus calculation
result = sp.calculate(
    prepared_qty=100.0,
    forecast_demand=82.0,
    item_name='Rice Bowl',
    unit='portions'
)
print('=== Single Surplus Example ===')
for k, v in result.items():
    if v is not None:
        print('  %-25s: %s' % (k, v))

In [ ]:
# End-to-end: forecast -> surplus for 7 weeks
forecasts = fc.forecast_next_n_weeks(55, 1885, n=7)

# Simulate kitchen prepares 15% more than forecast
prepared_list = [r['predicted_demand'] * 1.15 for r in forecasts]
batch_results = sp.batch_calculate(prepared_list, forecasts, unit='portions')

print('=== End-to-End Surplus Forecast (15% Over-Preparation) ===')
print('%-6s %-12s %-12s %-12s %-8s %-8s' % (
    'Week', 'Prepared', 'Forecast', 'Surplus', 'Pct%', 'Risk'))
print('-' * 65)
for r in batch_results:
    print('%-6s %-12.1f %-12.1f %-12.1f %-8.1f %-8s' % (
        r['week'], r['prepared_qty'], r['forecast_demand'],
        r['estimated_surplus'], r['surplus_pct'], r['risk_level']))

summary = sp.summarise_batch(batch_results)
print()
print('=== Batch Summary ===')
for k, v in summary.items():
    print('  %-25s: %s' % (k, v))

## 6. Known Limitations

1. **Integer weeks, not calendar dates** — The dataset uses week numbers 1–145. There are no actual calendar dates. Cyclical sin/cos features approximate within-year seasonality but cannot model specific Indian holidays automatically.

2. **Event calendar requires manual input** — Events (Diwali, exam periods, etc.) must be provided as a list of dicts with week numbers and multipliers. The model does not auto-detect festivals.

3. **Cross-validation uses a 50K row subsample** — The full 456K row dataset is sampled for CV metrics to keep training time reasonable. The final model trains on all rows.

4. **RF CV MAE (~119) > Baseline MAE (~113)** — This occurs because the baseline (mean per center-meal pair) is computed on the same training data. Out-of-sample RF performance is expected to be better. The CV subsample introduces some variance.

5. **No lag features** — The dataset rows are not guaranteed to be ordered temporally per center-meal, making standard lag features unreliable. Cyclical week features capture seasonality instead.

6. **Surplus engine is arithmetic, not ML** — The surplus predictor does not learn from historical over-preparation. A configurable `op_factor` can be set if kitchen-level over-preparation data becomes available.


## 7. Integration Interface for app.py

```python
from models.demand_forecast import DemandForecaster
from models.surplus_predictor import SurplusPredictor

# --- Demand forecasting ---
fc = DemandForecaster(n_estimators=200)
fc.train()   # loads datasets automatically

# Predict specific weeks
forecasts = fc.predict(
    center_id=55,
    meal_id=1885,
    weeks=[146, 147, 148, 149, 150, 151, 152],
    events=None   # or list of {week, event_name, event_type, multiplier}
)
# Returns: list of dicts with week, predicted_demand, lower_bound, upper_bound, std_dev

# Convenience: next N weeks from last training week
forecasts = fc.forecast_next_n_weeks(center_id=55, meal_id=1885, n=7)

# Metrics
metrics = fc.get_metrics()  # cv_mae, cv_rmse, baseline_mae, baseline_rmse

# --- Surplus prediction ---
sp = SurplusPredictor(low_threshold=10.0, high_threshold=25.0)

# Single calculation
result = sp.calculate(prepared_qty=120.0, forecast_demand=100.0, item_name='Rice', unit='kg')
# Returns: prepared_qty, forecast_demand, estimated_surplus, surplus_pct, risk_level, explanation

# Batch (one per forecast week)
prepared_list = [f['predicted_demand'] * 1.15 for f in forecasts]
batch = sp.batch_calculate(prepared_list, forecasts)
summary = sp.summarise_batch(batch)
# summary: total_prepared, total_forecast, total_surplus, overall_surplus_pct, overall_risk
```
